# Agentic Report

### No Claude Code Used!

In [1]:
from typing import TypedDict, List, Optional
from langchain_core.tools import tool, InjectedToolArg
from langgraph.graph import StateGraph, END
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from dotenv import load_dotenv
import PyPDF2
import os
import openai
import warnings
import re
from book_agent import book_lookup
from python_tool import execute_python_code
from prompts import *
import pandas as pd
from langgraph.prebuilt import ToolNode
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage, ChatMessage

warnings.filterwarnings('ignore')
print("✅ All libraries imported successfully!")

✅ Helper functions with citation support defined!
✅ All libraries imported successfully!


In [2]:
# Option 1: Load from .env file
load_dotenv()

# Verify API key is set
if os.getenv("OPENAI_API_KEY"):
    print("✅ API key loaded successfully!")
else:
    print("❌ API key not found. Please set OPENAI_API_KEY")

✅ API key loaded successfully!


In [3]:
class AgentState(TypedDict):
    task: str
    data: pd.DataFrame
    messages: str

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.utils.function_calling import convert_to_openai_function

tools = [book_lookup, execute_python_code]
functions = [convert_to_openai_function(f) for f in tools]

model = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2).bind(functions=functions)

In [5]:
def load_data_node(AgentState):
    """
    Agent loads data and gets comfortable with what it is looking at. Creates_graphs as needed
    """
    messages = AgentState.get("messages", [])
    if len(messages) > 1:
        print(" Tools already executed, ending")
        return state

    print(" First run - loading data and generating code")
    file_path = "aggregated_features_addresses.csv"
    
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
    else:
        raise FileNotFoundError(f"{file_path} not found")

    response = model.invoke([
        SystemMessage(content=LOAD_DATA_PROMPT),
        HumanMessage(content=f"Here's the data:n{df.head()}\\nColumns: {df.columns.tolist()}\n\nWrite code to generate relevant graphs of this data and save them as PNG files.")
    ])

    return {"messages":messages + [response],
           "data": df
           }

In [6]:
def organize_task_node(AgentState):
    """
    Agent takes in task and data and figures out what is needed to write the report.
    Comes up with query to ask the book.
    """

In [7]:
def write_report(AgentState):
    """
    Agent writes the report
    """

In [8]:
# Add nodes
agent_workflow = StateGraph(AgentState)

agent_workflow.add_node("load_data", load_data_node) ####
agent_workflow.add_node("tools", ToolNode([execute_python_code]))
#agent_workflow.add_node("organize_task", organize_task_node) ###
#agent_workflow.add_node("research", research_node) ####
#agent_workflow.add_node("write_report", write_report_node) ####

# graph structure
agent_workflow.set_entry_point("load_data")
def should_continue(state):
    last_message = state["messages"][-1]
    if last_message.tool_calls or last_message.additional_kwargs.get("function_call"):
        return "tools"
    return END
    
agent_workflow.add_conditional_edges(
    "load_data",
    should_continue
)
agent_workflow.add_edge("tools","load_data") #"organize_task"
#agent_workflow.add_edge("load_data","research")
#agent_workflow.add_edge("research","write_report")
#agent_workflow.add_edge("load_data", END) # write_report
graph = agent_workflow.compile()

In [9]:
result = graph.invoke({
    "messages": []})

 First run - loading data and generating code
 First run - loading data and generating code



KeyboardInterrupt



In [ ]:
result['messages'][-1]